In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random

# set random seed for reproducibility
random.seed(42)
np.random.seed(42)


# plot resolution and rendering
plt.rcParams["figure.dpi"] = 160
plt.rcParams["text.usetex"] = True
plt.rcParams["text.latex.preamble"] = r"\usepackage{{amsmath}}"
plt.style.use("dark_background")

# paths
INPUT_DATA = "../../../data/interpolation/auswertung_gesamt_10N_original.csv"

# Paramter
signal_variance = 0.8 # sigma_f^2
length_scale = 0.8 # l
noise_variance = 4e-4 # sigma_n^2

In [87]:
raw_data = pd.read_csv(INPUT_DATA)
raw_data = raw_data.drop(columns=["Datei_ID"])

data_filtered = raw_data.iloc[:, -8:]
mins = data_filtered.min()
maxs = data_filtered.max()
norm_data_filtered = (data_filtered - mins) / (maxs - mins)
norm_data_filtered = 2 * norm_data_filtered - 1
norm_data = raw_data.copy()
norm_data.iloc[:, -8:] = norm_data_filtered
norm_data.head()

# for testing we extract one random sample from the dataset
random_index = random.randint(0, norm_data.shape[0] - 1)
test_sample = norm_data.iloc[random_index]
print(f"Randomly selected test sample (index {random_index}): {test_sample.tolist()}")

# remove the test sample from the dataset
norm_data = norm_data.drop(index=random_index).reset_index(drop=True)


Randomly selected test sample (index 81): [116.0, 58.0, 10.0, 0.8754572663784506, 0.2339205243752558, 0.6970138383102698, 0.3281160838020707, 0.6128707893413776, -0.15497620525553457, 0.77805776313239, 0.27292340884573796]


In [88]:
# Data Matrix
X = norm_data.iloc[:, -8:].values
print(f"Data Matrix Shape: {X.shape}") # (121, 8)

# Labels
Y_x = norm_data.iloc[:, 0].values
Y_y = norm_data.iloc[:, 1].values
Y_F = norm_data.iloc[:, 2].values
print(f"Labels Shape: {Y_x.shape}, {Y_y.shape}, {Y_F.shape}") # (121,), (121,), (121,)

Data Matrix Shape: (120, 8)
Labels Shape: (120,), (120,), (120,)


In [89]:
def RBF_kernel(X1, X2, signal_variance=1.0, length_scale=1.0):
    """
    Computes the RBF kernel matrix between two sets of input points.

    Parameters:
    X1: (N, 8) data matrix
    X2: (N, 8) data matrix
    signal_variance: scalar, the variance of the signal (sigma_f^2)
    length_scale: scalar, the length scale of the kernel (l)

    Returns:
    K: (N, N) kernel matrix
    """
    K = np.zeros((X1.shape[0], X2.shape[0])) # (N, N)

    for i in range(X1.shape[0]):
        for j in range(X2.shape[0]):
            diff = X1[i] - X2[j] # (8,)
            sqdist = np.sum(diff**2) # scalar
            K[i, j] = signal_variance * np.exp(-0.5 * sqdist / length_scale**2)
    return K

# Compute the kernel matrix for the training data
K = RBF_kernel(X, X, signal_variance, length_scale) # (121, 121)

In [96]:
# Prediction for the test sample
X_test = test_sample[-8:].values.reshape(1, -1) # (1, 8)
k_star = RBF_kernel(X, X_test, signal_variance, length_scale)  # Kernelvector # (120, 1)
k_star_star = RBF_kernel(X_test, X_test, signal_variance, length_scale) # Scalar
inverse = np.linalg.inv(K + noise_variance * np.identity(K.shape[0])) # (120, 120)

# Posterior mean and variance for x
mu_star = k_star.T @ inverse @ Y_x # (1, 120) @ (120, 120) @ (120,) -> scalar
sigma_square_star = k_star_star - k_star.T @ inverse @ k_star # (1, 1) - (1, 120) @ (120, 120) @ (120, 1) -> scalar
print(f"Predicted mean for x: {mu_star}")
print(f"Predicted variance for x: {sigma_square_star}")

# Posterior mean and variance for y
mu_star_y = k_star.T @ inverse @ Y_y # (1, 120) @ (120, 120) @ (120,) -> scalar
sigma_square_star_y = k_star_star - k_star.T @ inverse @ k_star # (1, 1) - (1, 120) @ (120, 120) @ (120, 1) -> scalar
print(f"Predicted mean for y: {mu_star_y}")
print(f"Predicted variance for y: {sigma_square_star_y}")

# Posterior mean and variance for F
mu_star_F = k_star.T @ inverse @ Y_F # (1, 120) @ (120, 120) @ (120,) -> scalar 
sigma_square_star_F = k_star_star - k_star.T @ inverse @ k_star # (1, 1) - (1, 120) @ (120, 120) @ (120, 1) -> scalar
print(f"Predicted mean for F: {mu_star_F}")
print(f"Predicted variance for F: {sigma_square_star_F}")


Predicted mean for x: [112.57115417]
Predicted variance for x: [[0.00065616]]
Predicted mean for y: [52.3586236]
Predicted variance for y: [[0.00065616]]
Predicted mean for F: [10.06092636]
Predicted variance for F: [[0.00065616]]
